# Deep Learning and Anomaly Detection for Digital Behavior Fingerprinting
This notebook implements and evaluates:
1. A PyTorch Feed-Forward Neural Network (MLP) for multi-class subject identification.
2. A PyTorch Autoencoder for behavioral anomaly detection using reconstruction error thresholding.


In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.config import ALL_FEATURE_COLUMNS, TRAIN_SESSIONS, TEST_SESSIONS, RANDOM_SEED
from src.preprocessing import load_raw_dataset, session_based_split, fit_scaler, transform_features, fit_label_encoder
from src.features import extract_features
from src.models import KeystrokeMLP, train_neural_network, KeystrokeAutoencoder, train_autoencoder, detect_anomalies
from src.evaluation import evaluate_anomaly_detection

print(f'PyTorch version: {torch.__version__}')


## 1. Prepare Features and Session-Split


In [ ]:
df_raw = load_raw_dataset('../data/raw/DSL-StrongPasswordData.csv')
df_features = extract_features(df_raw)

train_df, test_df = session_based_split(
    df_features, train_sessions=TRAIN_SESSIONS, test_sessions=TEST_SESSIONS
)

X_train_df = train_df[ALL_FEATURE_COLUMNS]
X_test_df = test_df[ALL_FEATURE_COLUMNS]

scaler = fit_scaler(X_train_df)
X_train_scaled = transform_features(scaler, X_train_df, ALL_FEATURE_COLUMNS)
X_test_scaled = transform_features(scaler, X_test_df, ALL_FEATURE_COLUMNS)

label_encoder = fit_label_encoder(train_df['subject'].values)
y_train = label_encoder.transform(train_df['subject'].values)
y_test = label_encoder.transform(test_df['subject'].values)
num_classes = len(label_encoder.classes_)

print(f'X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}, Classes: {num_classes}')


## 2. Train PyTorch Neural Network (MLP) for Subject Classification


In [ ]:
mlp_model, mlp_metrics = train_neural_network(
    X_train=X_train_scaled,
    y_train=y_train,
    X_val=X_test_scaled,
    y_val=y_test,
    num_classes=num_classes,
    epochs=35,
    batch_size=128,
    learning_rate=1e-3,
    seed=RANDOM_SEED
)

print('PyTorch Neural Network Evaluation:')
for metric_name, val in mlp_metrics.items():
    print(f'  {metric_name}: {val:.4f}')


## 3. Train PyTorch Autoencoder for Anomaly Detection
The autoencoder is trained on normal keystroke features to learn compact latent representations.
Reconstruction error measures deviation from familiar behavioral patterns.


In [ ]:
autoencoder, threshold, ae_metrics = train_autoencoder(
    X_train=X_train_scaled,
    epochs=35,
    batch_size=128,
    threshold_percentile=95.0,
    seed=RANDOM_SEED
)

print(f'Calibrated Anomaly Threshold: {threshold:.5f}')
print(f'Training Reconstruction Error Mean: {ae_metrics["mean_reconstruction_error"]:.5f}')


## 4. Anomaly Detection Evaluation: Legitimate vs Imposter Patterns


In [ ]:
# Normal test errors
test_errors, _ = detect_anomalies(autoencoder, X_test_scaled, threshold)

# Simulated imposter errors (perturbed typing speeds and key latencies)
rng = np.random.RandomState(RANDOM_SEED)
noise = rng.normal(loc=1.5, scale=1.0, size=X_test_scaled.shape)
X_imposter_scaled = X_test_scaled + noise
imposter_errors, _ = detect_anomalies(autoencoder, X_imposter_scaled, threshold)

anomaly_results = evaluate_anomaly_detection(test_errors, imposter_errors, threshold)
print(f'ROC-AUC: {anomaly_results["roc_auc"]:.4f}')
print(f'True Positive Rate (Imposter Detection): {anomaly_results["tpr_detection_rate"] * 100:.1f}%')
print(f'False Positive Rate (False Alarms): {anomaly_results["fpr_false_alarm_rate"] * 100:.1f}%')


## 5. Visualize Reconstruction Error Distribution


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(test_errors, bins=50, alpha=0.7, label='Normal Keystroke Patterns (Test Sessions)', color='steelblue', density=True)
plt.hist(imposter_errors, bins=50, alpha=0.6, label='Imposter/Anomalous Patterns', color='crimson', density=True)
plt.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.4f})')
plt.title('Autoencoder Reconstruction Error Distribution & Anomaly Threshold')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()
